<a href="https://colab.research.google.com/github/hbprosper/dihiggs/blob/main/code/heft_train_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Higgs Effective Field Theory (HEFT) Study: Bootstrap Ensemble
> Created: Oct 30, 2026 Harrison B. Prosper and Claude<br>

## Introduction

`heft_train.ipynb` trains *one* `HEFTNet` model. Here we train an **ensemble** of them and use the spread of their predictions as an estimate of the **standard error** of the trained network. The plan is described in `docs/ensemble/ensemble_plan.pdf`.

### The estimator view

Think of the whole training procedure as an *estimator*: it takes a training set $D$ and a random seed $s$ (which fixes the initial weights and the order of the batches) and returns a function $\hat f(\boldsymbol{x}; D, s)$. Had the training set been a different draw from the same population, or the seed different, the result would differ. At a fixed point $\boldsymbol{x}$, $\hat f(\boldsymbol{x})$ is therefore a random variable; its standard deviation is the standard error of the trained network.

### The bootstrap

We have one training set, so we cannot observe the spread of $\hat f$ over training sets directly. The **bootstrap** (Efron, 1979) replaces the unknown population $P$ from which the $N$ training rows were drawn by the *empirical* distribution of those rows, which puts probability $1/N$ on each. A bootstrap training set is a sample of $N$ rows drawn from the training rows **with replacement**: on average it contains $1 - e^{-1} \approx 63\%$ of the distinct rows, some of them several times. The spread of the networks trained on many bootstrap sets estimates the spread over training sets drawn from $P$.

### Two ensembles

| `MODE` | training rows of member $m$ | seed | estimates |
| :--- | :--- | :--- | :--- |
| `'seeds'` | the 274,152 training rows | `SEED + m` | the optimisation spread: which valley |
| `'bootstrap'` | 274,152 rows drawn from them | `SEED + m` | the total spread |

By the law of total variance,
$$
\text{Var}_{D,s}[\hat f] = \text{E}_D\big[\text{Var}_s[\hat f \mid D]\big] + \text{Var}_D\big[\text{E}_s[\hat f \mid D]\big],
$$
the total (bootstrap ensemble) is the optimisation term (seeds ensemble) plus a data term, which is their difference. Member $m$ uses the same seed in both ensembles, so the two members start from the same initial weights and differ only in their training rows.

### The reference ensemble

Every standard error refers to an imagined ensemble of repetitions. Here the repetitions are new, independent draws of the training rows from the population $P$ of rows, as the pipeline sees them. Other choices are possible (new HEFT points; new POWHEG events at the same points), and they answer different questions; see Section 3 of the plan. Whatever the choice, the test at the end of this notebook, on the holdout spectra, checks whether the spread matches the actual errors.

### What the spread does not capture

All members share the architecture, the loss and the training schedule. Where they all make the same error, for example near threshold, they agree, and the spread is small. The pull test at the end measures how much of the actual error the spread accounts for.

## Bibliography
  1. B. Efron, Ann. Statist. 7 (1979) 1.
  2. B. Efron and R. J. Tibshirani, *An Introduction to the Bootstrap*, Chapman & Hall/CRC (1993).
  3. B. Lakshminarayanan, A. Pritzel and C. Blundell, NeurIPS 2017, arXiv:1612.01474.

In [ ]:
# ------------------------------------------------------------------
# Google Colab setup (nothing happens when run outside Colab)
#   1. clone the dihiggs repository, which holds the data files,
#   2. install the dihiggs package, which also installs drwho,
#   3. make dihiggs/code the working directory so that the relative
#      paths used below (../data, runs/) work as they do locally.
#
# Training is much faster on a GPU: in Colab choose
#   Runtime > Change runtime type > GPU.
# ------------------------------------------------------------------
import os, sys, subprocess, importlib

IN_COLAB = 'google.colab' in sys.modules

# With USE_DRIVE = True the runs/ folder lives in your Google Drive
# (MyDrive/dihiggs/runs), which keeps the members safe if the Colab
# runtime disconnects. You will be asked to allow access to your Drive.
USE_DRIVE = True

if IN_COLAB:
    REPO = '/content/dihiggs'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/hbprosper/dihiggs.git', REPO],
                       check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', REPO],
                   check=True)
    importlib.invalidate_caches()
    os.chdir(f'{REPO}/code')

    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_runs = '/content/drive/MyDrive/dihiggs/runs'
        os.makedirs(drive_runs, exist_ok=True)
        if not os.path.exists('runs'):
            os.symlink(drive_runs, 'runs')
        print(f'runs/ is stored in {drive_runs}')

print(f'working directory: {os.getcwd()}')

In [ ]:
import os, sys, time, csv

# the standard module for tabular data
import pandas as pd

# the standard module for array manipulation
import numpy as np

# the standard modules for high-quality plots
import matplotlib as mp
import matplotlib.pyplot as plt

# pytorch
import torch
import torch.nn as nn

# module for shell utilities
import shutil

# ML and other utilities
import drwho.nn as mlp
import drwho.data as dat
import drwho.monitor as mon

# the ensemble of trained networks
from dihiggs import ensemble

# update fonts
plt.rcParams.update({
  "text.usetex": shutil.which('latex') is not None,
  "font.family": "sans-serif",
  "font.size": 12
})

## Computational device

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n\tAvailable device: {str(DEVICE):4s}\n')

## Load training data

In [ ]:
datafile = f'../data/heft_wide_gauss_traindata_13TeV.csv.gz'

print(f'loading {datafile}')
df = pd.read_csv(datafile)

sample_size = len(df)

# Fingerprint of the training file, saved in each member's configuration.
import hashlib
with open(datafile, 'rb') as f:
    traindata_sha256 = hashlib.sha256(f.read()).hexdigest()
print(f'sample size: {sample_size}')

## Switches

| switch | meaning |
| :--- | :--- |
| `MODE` | `'bootstrap'` (resample the training rows) or `'seeds'` (the training rows as they are) |
| `N_MEMBERS` | size of the ensemble |
| `MEMBERS` | which members *this* notebook trains, e.g. `range(0, 10)`; two copies of the notebook with disjoint ranges can train at the same time |
| `SEED` | member $m$ uses seed `SEED + m` for its initial weights and batch order |
| `BOOT_SEED` | member $m$ draws its bootstrap rows with `np.random.RandomState(BOOT_SEED + m)` |
| `N_ITERATIONS` | training iterations per member (200,000 in `heft_train.ipynb`; fewer for a first trial) |

A member whose parameter file and summary row already exist is skipped, so an interrupted loop resumes where it stopped.

In [ ]:
# ------------------------------------------------------------------
# Switches
# ------------------------------------------------------------------
MODE         = 'bootstrap'     # 'bootstrap' or 'seeds'
N_MEMBERS    = 20
MEMBERS      = range(N_MEMBERS)
SEED         = 42
BOOT_SEED    = 1000
N_ITERATIONS = 200_000

assert MODE in ('bootstrap', 'seeds'), f'unknown mode {MODE}'

NAME    = 'dihiggs'
DIRNAME = f'ENS_{MODE}'
BASE    = f'runs/{DIRNAME}'
os.makedirs(BASE, exist_ok=True)
print(f'ensemble folder: {BASE}')

## The split

Exactly that of `heft_train.ipynb`: the first 86.4% of the rows (already shuffled when the training file was made) for training, the next 1.6% for validation, the last 12% as the quick test. Only the training rows are resampled; the validation rows, which choose the saved parameters of every member, are the same for all members of both ensembles and are never copies of training rows.

In [ ]:
from dihiggs import heftnet as NN

X = df[NN.FEATURES].to_numpy(dtype=np.float32)
# Target: the POWHEG cross section per bin (pb), one column, shape [N, 1]
Y = df[[NN.TARGET]].to_numpy(dtype=np.float32)

train_size = int(0.864*sample_size)
test_size  = int(0.120*sample_size)
val_size   = sample_size - train_size - test_size

batch_size = 2048

# validation and quick-test data: the same for every member
val_data  = dat.Dataset(X, targets=Y, start=train_size,
                        end=train_size + val_size)
test_data = dat.Dataset(X, targets=Y, start=train_size + val_size,
                        end=train_size + val_size + test_size)
val_loader  = dat.DataLoader(val_data,  batch_size=len(val_data))
test_loader = dat.DataLoader(test_data, batch_size=len(test_data))

print(f'train: {train_size}   val: {val_size}   test: {test_size}')

## One member's training rows

For `'seeds'`, all training rows. For `'bootstrap'`, `train_size` row numbers drawn uniformly, with replacement, from `0, ..., train_size - 1`. `np.random.RandomState` is used because its sequence of random numbers is frozen across numpy versions, so a member can be reproduced exactly. The function also returns how many times each training row was drawn (the "counts"); rows with count 0 are the member's *out-of-bag* rows, which it never sees.

In [ ]:
def member_rows(m):
    """
    Return (rows, counts) for member m (counting from 0):
      rows   - the row numbers of the member's training set, in the order
               used (a numpy array of length train_size),
      counts - how many times each of the train_size training rows appears
               (None for the 'seeds' ensemble, where every count is 1).
    """
    if MODE == 'seeds':
        return np.arange(train_size), None
    rng = np.random.RandomState(BOOT_SEED + m)
    # randint(low, high, size) draws integers in [low, high), i.e. low to
    # high-1, uniformly and independently: a draw with replacement
    rows = rng.randint(0, train_size, size=train_size)
    # bincount(rows) counts how often each row number occurs
    counts = np.bincount(rows, minlength=train_size)
    return rows, counts

# a quick look at member 0
rows, counts = member_rows(0)
if counts is not None:
    print(f'member 1: {np.sum(counts > 0)} distinct rows of {train_size} '
          f'({np.mean(counts > 0):.1%}); most repeated: {counts.max()} times')

## Training one member

The recipe of `heft_train.ipynb`: the L1 loss (mean absolute difference between the model and POWHEG), Adam, batch size 2,048, four learning-rate steps from $2 \times 10^{-3}$, the parameters saved whenever the validation loss improves. The real-time loss window of `heft_train.ipynb` is not opened (in a loop over 20 members it would open 20 windows); the losses are still written to each member's `dihiggs_losses.csv`.

Each member is saved in `runs/ENS_<mode>/member_NN/`, an ordinary run folder, so `heft_test.ipynb` and `heft_export.ipynb` work with any member. Bootstrap members also get `bootstrap_counts.npz`. One row per member is appended to `runs/ENS_<mode>/ensemble.csv`.

In [ ]:
class L1(nn.Module):
    """
    Average absolute difference between the predictions f (shape [B]) and
    the POWHEG cross sections in column 0 of the targets y (shape [B, 1]).
    This is the 'L1' loss of heft_train.ipynb.
    """
    def forward(self, f, y):
        # view(-1) flattens a tensor to one dimension, so that f and y[:, 0]
        # have the same shape [B]
        return torch.mean(torch.abs(f.view(-1) - y[:, 0]))


def train(objective, optimizer, scheduler, monitor,
          train_loader, train_small_loader, val_loader):
    # the training loop of heft_train.ipynb
    for x, y in train_loader:
        objective.train()          # training mode
        optimizer.zero_grad()      # clear the gradients
        R = objective(x, y)        # empirical risk of this batch
        R.backward()               # its gradients
        optimizer.step()           # one step downhill
        scheduler.step()           # update the learning rate if it is time
        if monitor.step():
            t_loss = mlp.compute_avg_loss(objective, train_small_loader)
            v_loss = mlp.compute_avg_loss(objective, val_loader)
            monitor(t_loss, v_loss, scheduler.lr())


def mean_abs_error(model, rows):
    """
    Average |model - POWHEG| over the given training rows (for the
    out-of-bag rows), computed in chunks to limit memory.
    """
    total, n = 0.0, 0
    model.eval()
    with torch.no_grad():
        for start in range(0, len(rows), 2**16):
            r = rows[start:start + 2**16]
            x = torch.from_numpy(X[r]).to(DEVICE)
            t = torch.from_numpy(Y[r, 0]).to(DEVICE)
            total += float(torch.sum(torch.abs(model(x).view(-1) - t)))
            n += len(r)
    return total / max(n, 1)


SUMMARY = f'{BASE}/ensemble.csv'
SUMMARY_COLUMNS = ['member', 'seed', 'boot_seed', 'n_rows', 'n_distinct',
                   'best_val_loss', 'oob_loss', 'minutes']

def finished_members():
    # the member numbers (1, 2, ...) already in the summary file
    if not os.path.exists(SUMMARY):
        return set()
    return set(int(m) for m in pd.read_csv(SUMMARY)['member'])


def train_member(m):
    """
    Train member m (counting from 0); it is saved as member_{m+1:02d}.
    """
    t0 = time.time()
    member = m + 1
    dirname = f'{DIRNAME}/member_{member:02d}'

    # 1. seeds: initial weights, batch order and the random subset used to
    #    monitor the training loss all depend on SEED + m
    torch.manual_seed(SEED + m)
    np.random.seed(SEED + m)

    # 2. this member's training rows
    rows, counts = member_rows(m)
    Xm, Ym = X[rows], Y[rows]

    # 3. configuration, as in heft_train.ipynb, plus the ensemble settings
    config = mlp.Config(NAME, dirname=dirname)
    config('model_name', NN.NAME)
    config('features', NN.FEATURES)
    config('target', NN.TARGET)
    config('activation', NN.ACTIVATION)
    config('traindata_sha256', traindata_sha256)
    config('loss', 'L1')
    config('batch_size', batch_size)
    config('train_size', train_size)
    config('test_size', test_size)
    config('val_size', val_size)
    config('monitor_step', 100)
    config('frac', 0.015)
    n_steps = 4
    config('n_steps', n_steps)
    config('n_iterations', N_ITERATIONS)
    config('n_iters_per_step', int(N_ITERATIONS / n_steps))
    config('base_lr', 2.e-3)
    config('gamma', 0.5)
    config('ensemble/mode', MODE)
    config('ensemble/member', member)
    config('ensemble/seed', SEED + m)
    config('ensemble/boot_seed', BOOT_SEED + m if MODE == 'bootstrap' else -1)
    config.save()
    if counts is not None:
        # savez_compressed writes numpy arrays to a compressed .npz file
        np.savez_compressed(f'runs/{dirname}/bootstrap_counts.npz',
                            counts=counts.astype(np.uint8))

    # 4. data loaders for this member
    train_data = dat.Dataset(Xm, targets=Ym, start=0, end=len(Xm), verbose=0)
    train_data_val = dat.Dataset(Xm, targets=Ym, start=0, end=len(Xm),
                                 random_sample_size=val_size, verbose=0)
    train_loader = dat.DataLoader(train_data, batch_size=batch_size,
                                  num_iterations=N_ITERATIONS, verbose=0)
    train_loader_val = dat.DataLoader(train_data_val,
                                      batch_size=len(train_data_val),
                                      verbose=0)

    # 5. model, objective, optimiser, scheduler, monitor
    model = NN.HEFTNet().to(DEVICE)
    objective = mlp.Objective(model, L1())
    optimizer = torch.optim.Adam(model.parameters(), lr=config('base_lr'))
    scheduler = mlp.LRStepScheduler(optimizer,
                                    n_steps=config('n_steps'),
                                    n_iters_per_step=config('n_iters_per_step'),
                                    base_lr=config('base_lr'),
                                    gamma=config('gamma'), verbose=False)
    monitor = mon.Monitor(N_ITERATIONS, config('file/losses'),
                          monitorstep=config('monitor_step'),
                          newlossfile=True, frac=config('frac'),
                          model=model, paramsfile=config('file/params'))

    # 6. train (the real-time window is not started)
    train(objective, optimizer, scheduler, monitor,
          train_loader, train_loader_val, val_loader)
    print()

    # 7. reload the best parameters; validation and out-of-bag losses
    model = NN.HEFTNet().to(DEVICE)
    model.load(config('file/params'))
    model.to(DEVICE)
    best_val = mlp.compute_avg_loss(mlp.Objective(model, L1()), val_loader)
    if counts is not None:
        oob = mean_abs_error(model, np.where(counts == 0)[0])
        n_distinct = int(np.sum(counts > 0))
    else:
        oob, n_distinct = float('nan'), train_size

    # 8. one row in the summary file
    minutes = (time.time() - t0) / 60
    new = not os.path.exists(SUMMARY)
    with open(SUMMARY, 'a', newline='') as f:
        writer = csv.writer(f)
        if new:
            writer.writerow(SUMMARY_COLUMNS)
        writer.writerow([member, SEED + m,
                         BOOT_SEED + m if MODE == 'bootstrap' else -1,
                         len(rows), n_distinct, f'{best_val:.6g}',
                         f'{oob:.6g}', f'{minutes:.2f}'])
    print(f'member {member:2d}: best validation loss {best_val:.4e}, '
          f'out-of-bag loss {oob:.4e}, {minutes:.1f} min')

## Train the ensemble

Members already finished (parameter file present and listed in `ensemble.csv`) are skipped. After the first member, the time for the rest is estimated.

In [ ]:
TRAIN = True

if TRAIN:
    done = finished_members()
    todo = [m for m in MEMBERS
            if not (m + 1 in done and
                    os.path.exists(f'{BASE}/member_{m+1:02d}/{NAME}_params.pth'))]
    print(f'{len(todo)} members to train: {[m + 1 for m in todo]}')
    t_start = time.time()
    for k, m in enumerate(todo):
        print(f'\n--- member {m + 1} of {N_MEMBERS} ---')
        train_member(m)
        per_member = (time.time() - t_start) / (k + 1)
        left = per_member * (len(todo) - k - 1) / 60
        print(f'    about {left:.0f} min left for this notebook')

## Results on the holdout spectra

The ensemble is read with `ensemble.Ensemble`, which, like `hmc.BNN`, computes the mean, standard deviation, quantiles and intervals of the members' predictions at any set of points.

### The POWHEG noise

As in `heft_test.ipynb` and `heft_train_bnn.ipynb`: each spectrum behaves as a histogram of about $N_\text{eff}$ equal-weight events, so the noise variance of a bin with cross section $t$ in a spectrum of total cross section $\sigma_\text{tot}$ is $\sigma_k^2 = t\,\sigma_\text{tot} / N_\text{eff}$. $N_\text{eff}$ is estimated from the training spectra.

In [ ]:
# N_eff from the bin-to-bin jitter of the training spectra
spectra_train = df.pivot_table(index='ID', columns='mhh', values='sigma',
                               fill_value=0.0)
F     = spectra_train.to_numpy(np.float64)
F_tot = F.sum(axis=1, keepdims=True)
f_mid, f_lo, f_hi = F[:, 1:-1], F[:, :-2], F[:, 2:]
D = f_mid - (f_lo + f_hi) / 2
good = (f_mid > 0) & (f_lo > 0) & (f_hi > 0)
q = np.full_like(D, np.nan)
q[good] = D[good]**2 / (F_tot * (f_mid + (f_lo + f_hi) / 4))[good]
smooth = spectra_train.columns.to_numpy()[1:-1] >= (38 - 17 + 0.5) / 100
N_EFF  = 0.4549 / np.nanmedian(q[:, smooth])
print(f'effective number of events per spectrum: N_eff = {N_EFF:.0f}')

# the holdout spectra: one row per spectrum, one column per bin
spectra = pd.read_csv('../data/heft_wide_gauss_testdata_13TeV.csv')
params  = ['klambda', 'CT', 'CTT', 'CGGH', 'CGGHH']
bins    = [c for c in spectra.columns if c.isdigit()]
T       = spectra[bins].to_numpy(np.float64)          # [1000, 80] POWHEG
n_spec, n_bins = T.shape
x_bins  = (np.arange(n_bins) + 0.5) / 100             # bin centres in x
mhh     = 255 + 1500 * x_bins                         # in GeV

# the model inputs for every bin of every holdout spectra: [80000, 6]
theta = spectra[params].to_numpy(np.float32)
inputs = np.column_stack([np.tile(x_bins, n_spec).astype(np.float32),
                          np.repeat(theta, n_bins, axis=0)])
x_hold = torch.from_numpy(inputs).to(DEVICE)

# the POWHEG noise variance of each holdout bin (NaN for empty bins)
T_tot = T.sum(axis=1, keepdims=True)
noise_var = np.where(T > 0, T * T_tot / N_EFF, np.nan)

### Accuracy, width and calibration

* **Accuracy:** $\chi^2/N$ of each spectrum, $\chi^2 = \sum_k (t_k - \bar f_k)^2/\sigma_k^2$, for the ensemble mean and for each member.
* **Width:** the ensemble standard deviation $s_k$ in units of the POWHEG noise $\sigma_k$.
* **Calibration:** the pull $z_k = (t_k - \bar f_k)/\sqrt{\sigma_k^2 + s_k^2}$. If the spread accounts for the model's error, $z$ has standard deviation 1 and $|z| < 2$ for about 95% of bins.

In [ ]:
ens = ensemble.Ensemble(NN.HEFTNet, BASE, device=DEVICE)
K = len(ens)
print(f'{K} members in {BASE}')

# all members' predictions: [K, 1000, 80]
P = ens.predict(x_hold).double().cpu().numpy().reshape(K, n_spec, n_bins)
mean_f = P.mean(axis=0)
sd_f   = P.std(axis=0, ddof=1)        # ddof=1: divide by K - 1

def chi2_per_bin(f):
    # (t - f)^2 / sigma^2 for every holdout bin; NaN for empty bins
    return (T - f)**2 / noise_var

def summary(c):
    # c: [1000, 80] chi-squared per bin
    c_spec = np.nanmean(c, axis=1)                    # chi2/N per spectrum
    return (np.median(c_spec), np.mean(c_spec > 2), np.mean(c_spec > 5),
            np.nanmean(c[:, :20]), np.nanmean(c[:, 20:]))

rows_out = [('ensemble mean',) + summary(chi2_per_bin(mean_f))]
member_stats = np.array([summary(chi2_per_bin(P[k])) for k in range(K)])
rows_out.append(('single member (median)',) + tuple(np.median(member_stats, axis=0)))

print(f'{"":24s} {"median chi2/N":>14s} {">2":>7s} {">5":>7s} '
      f'{"255-555 GeV":>12s} {">555 GeV":>9s}')
for name, med, f2, f5, lo, hi in rows_out:
    print(f'{name:24s} {med:14.3f} {f2:7.1%} {f5:7.1%} {lo:12.2f} {hi:9.2f}')

ratio = sd_f / np.sqrt(noise_var)
z = (T - mean_f) / np.sqrt(noise_var + sd_f**2)
ok = np.isfinite(z)
print(f'spread / POWHEG noise: median {np.nanmedian(ratio):.3f} '
      f'(255-555 GeV {np.nanmedian(ratio[:, :20]):.3f}, '
      f'above {np.nanmedian(ratio[:, 20:]):.3f})')
print(f'pull: s.d. {np.std(z[ok]):.3f} '
      f'(255-555 GeV {np.nanstd(z[:, :20]):.3f}, '
      f'above {np.nanstd(z[:, 20:]):.3f}); '
      f'|z| < 2 for {np.mean(np.abs(z[ok]) < 2):.1%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(mhh, np.nanmean(chi2_per_bin(mean_f), axis=0), 'b', label='ensemble mean')
ax.plot(mhh, np.nanmedian([np.nanmean(chi2_per_bin(P[k]), axis=0)
                           for k in range(K)], axis=0),
        'r', label='single member (median)')
ax.axhline(1, color='gray', ls=':')
ax.set_yscale('log')
ax.set_xlabel(r'$m_{hh}$ (GeV)')
ax.set_ylabel(r'mean $\chi^2$ per bin')
ax.legend(fontsize=10)

ax = axes[1]
ax.plot(mhh, np.nanmedian(ratio, axis=0), 'b')
ax.set_yscale('log')
ax.set_xlabel(r'$m_{hh}$ (GeV)')
ax.set_ylabel('spread / POWHEG noise (median)')

ax = axes[2]
ax.plot(mhh, np.nanstd(z, axis=0), 'b', label='noise + spread')
ax.plot(mhh, np.nanstd((T - mean_f) / np.sqrt(noise_var), axis=0), 'b--',
        label='noise only')
ax.axhline(1, color='gray', ls=':')
ax.set_xlabel(r'$m_{hh}$ (GeV)')
ax.set_ylabel(r'standard deviation of the pull $z$')
ax.legend(fontsize=10)

fig.tight_layout()
plt.savefig(f'{BASE}/{NAME}_ensemble_holdout.png')
plt.show()

### Spectra with the ensemble band

A few holdout spectra with the ensemble mean and the band containing the central 68% of the members, and below, the band relative to the mean.

In [ ]:
rows_to_plot = [0, 1, 2, 3, 4, 5, 6, 7]
fig, axes = plt.subplots(4, 4, figsize=(13, 9),
                         gridspec_kw={'height_ratios': [3, 1, 3, 1]})
for n, row in enumerate(rows_to_plot):
    ax  = axes[2 * (n // 4), n % 4]
    axr = axes[2 * (n // 4) + 1, n % 4]
    # the members' predictions for this spectrum: [K, 80]
    p = P[:, row, :]
    lo, hi = np.quantile(p, [0.16, 0.84], axis=0)
    m = mean_f[row]
    ax.fill_between(mhh, lo, hi, color='steelblue', alpha=0.5, lw=0)
    ax.plot(mhh, m, 'b', lw=1)
    ax.step(mhh, T[row], 'k', where='mid', lw=0.7)
    ax.set_ylim(0, 1.3 * max(T[row].max(), hi.max()))
    ax.ticklabel_format(axis='y', scilimits=[-2, 2])

    # relative band, shown only where the mean is above 1% of its maximum
    scale = np.where(m > 0.01 * m.max(), m, np.nan)
    axr.fill_between(mhh, (lo - m) / scale, (hi - m) / scale,
                     color='steelblue', alpha=0.5, lw=0)
    axr.axhline(0, color='b', lw=0.8)
    # limit the vertical range so that a few bins near threshold do not
    # hide the rest of the band
    edges = np.concatenate([(lo - m) / scale, (hi - m) / scale])
    e_lo, e_hi = np.nanpercentile(edges, [10, 90])
    pad = 0.5 * (e_hi - e_lo)
    axr.set_ylim(e_lo - pad, e_hi + pad)
    axr.ticklabel_format(axis='y', scilimits=[-2, 2])
    if n % 4 == 0:
        ax.set_ylabel(r'$\sigma$ (pb)')
        axr.set_ylabel('rel.')
    axr.set_xlabel(r'$m_{hh}$ (GeV)')
fig.tight_layout()
plt.savefig(f'{BASE}/{NAME}_ensemble_spectra.png')
plt.show()

### The two sources of spread

When both ensembles exist, the law of total variance gives the data term as the difference between the bootstrap (total) variance and the seeds (optimisation) variance. Plotted are the square roots, in units of the POWHEG noise. The data term can come out slightly negative where the two are close, because each variance is estimated from a finite number of members (with 20 members, a standard deviation is uncertain by about 16%); it is set to zero there.

In [ ]:
other = {'bootstrap': 'runs/ENS_seeds', 'seeds': 'runs/ENS_bootstrap'}[MODE]
if os.path.exists(other):
    try:
        ens_other = ensemble.Ensemble(NN.HEFTNet, other, device=DEVICE)
    except FileNotFoundError:
        ens_other = None
else:
    ens_other = None

if ens_other is None:
    print(f'{other} not found: train the other ensemble to see the decomposition')
else:
    P2 = ens_other.predict(x_hold).double().cpu().numpy().reshape(
        len(ens_other), n_spec, n_bins)
    var_this, var_other = sd_f**2, P2.var(axis=0, ddof=1)
    var_total = var_this if MODE == 'bootstrap' else var_other
    var_opt   = var_other if MODE == 'bootstrap' else var_this
    var_data  = np.maximum(var_total - var_opt, 0.0)
    sigma = np.sqrt(noise_var)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for v, label, c in [(var_total, 'total (bootstrap)', 'k'),
                        (var_opt, 'optimisation (seeds)', 'b'),
                        (var_data, 'data (difference)', 'r')]:
        ax.plot(mhh, np.nanmedian(np.sqrt(v) / sigma, axis=0), color=c,
                label=label)
    ax.set_yscale('log')
    ax.set_xlabel(r'$m_{hh}$ (GeV)')
    ax.set_ylabel('standard deviation / POWHEG noise (median)')
    ax.legend(fontsize=10)
    fig.tight_layout()
    plt.savefig(f'{BASE}/{NAME}_ensemble_decomposition.png')
    plt.show()

    frac_opt = np.nanmedian(var_opt / np.where(var_total > 0, var_total, np.nan))
    print(f'median fraction of the total variance due to the optimisation: '
          f'{frac_opt:.2f}')

### The members' own record

From `ensemble.csv`: each member's best validation loss and, for bootstrap members, its loss on the out-of-bag rows, the training rows it never saw. The out-of-bag loss should be close to the validation loss; a much smaller value would mean the out-of-bag rows are not really "unseen", for example because neighbouring bins of the same spectrum were in the member's training set.

In [ ]:
info = ens.summary()
if info is not None:
    table = pd.DataFrame(info)
    print(table.to_string(index=False))
    if MODE == 'bootstrap':
        print(f'\nmedian out-of-bag / validation loss: '
              f'{np.median(table.oob_loss / table.best_val_loss):.3f}')